### Дообучение с небольшим числом обучаемых параметров (PEFT)

В этом ноутбуке вы будете дообучать большие языковые модели при ограниченном объёме видеопамяти. Вы реализуете prompt tuning и LoRA, а также попробуете библиотеку Hugging Face PEFT.

Для выполнения заданий нужен GPU с поддержкой CUDA. Замените заглушки вида `<YOUR CODE HERE>` своим кодом; в исходном виде ячейки с такими заглушками не выполняются. Английские фразы, которые подаются модели, оставлены без перевода: они являются данными для упражнений. Названия API и переменных также сохранены.


In [1]:
# Исходные версии библиотек
# %pip install --quiet transformers==4.34.1 accelerate==0.24.0 sentencepiece==0.1.99 optimum==1.13.2 peft==0.5.0 bitsandbytes==0.41.2.post2

# Версии, рекомендованные для Colab в октябре 2025 года (спасибо Льву!)
# %pip install --quiet "bitsandbytes==0.45.3" "transformers>=4.43,<4.46" "accelerate>=0.33,<0.36" "peft>=0.11.1" "optimum>=1.20.0" "sentencepiece"

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import transformers
from tqdm.auto import tqdm, trange
# assert torch.cuda.is_available(), "Для этой части задания нужен GPU с поддержкой CUDA"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [3]:
model_name = 'Enoch/llama-7b-hf'

# Загружаем токенизатор Llama...
tokenizer = transformers.LlamaTokenizer.from_pretrained(model_name, device_map=device)
tokenizer.pad_token_id = tokenizer.eos_token_id

# ...и саму модель
model = transformers.AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map='auto',
    low_cpu_mem_usage=True,
    offload_state_dict=True,
    load_in_4bit=True,
    torch_dtype=torch.float32,  # Веса — 4-битные; нормализации и активации — в fp32
)
for param in model.parameters():
    param.requires_grad=False

model.gradient_checkpointing_enable()  # Сохраняем лишь часть активаций, остальные вычисляем повторно
model.enable_input_require_grads()     # Для checkpointing включаем вычисление градиентов по входам, чтобы сохранить обратное распространение
# Подробнее о gradient checkpointing: https://pytorch.org/docs/stable/checkpoint.html https://arxiv.org/abs/1604.06174

You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama.LlamaTokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565 - if you loaded a llama tokenizer from a GGUF file you can ignore this message
The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/33 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/151 [00:00<?, ?B/s]

### Prompt tuning: история одной лисы (1 балл)

![Лиса из примера](https://i.imgur.com/Ux3qQAu.png) (источник: theodd1souts.fandom.com)


In [4]:
prompt = 'A quick brown fox'
batch = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False).to(device)

for i in range(10):
    next_token = model(**batch).logits[0, -1].argmax(-1).reshape(1, 1)
    batch['input_ids'] = torch.cat([batch['input_ids'], next_token], dim=-1)
    batch['attention_mask'] = torch.cat([batch['attention_mask'], torch.ones_like(next_token)], dim=-1)

print("\nРезультат:", tokenizer.decode(batch['input_ids'][0].cpu().numpy().tolist()))

Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)



Результат: <s>A quick brown fox jumps over the lazy dog.
A quick


Какая наглая ложь! Эта лиса уверяет, что вовсе не перепрыгивала через ленивую собаку. Она просто занималась своими делами. **Ваша задача — обучить модель говорить правду: сегодня через собаку никто не прыгал.**

В примере используется начало известной английской панграммы: «A quick brown fox» — «Быстрая бурая лиса». Целевая фраза `the_truth` отрицает привычное продолжение «jumped over the lazy dog». Для обучения используйте именно английские строки из ячеек ниже.


In [5]:
the_truth = "A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it anyway!"
batch = tokenizer(the_truth, return_tensors='pt', return_token_type_ids=False).to(device)
outputs = model(**batch)

next_word_logits = outputs.logits[:, :-1]
true_next_tokens = batch['input_ids'][:, 1:]
loss = F.cross_entropy(next_word_logits.flatten(0, 1), true_next_tokens.flatten(0, 1))

print("Потери:", loss)

Потери: tensor(3.0729, device='cuda:0', grad_fn=<NllLossBackward0>)


Однако обучать всю модель мы не можем: только градиенты в формате `float32` заняли бы около 28 ГБ. Вместо этого применим [prompt tuning](https://arxiv.org/abs/2104.08691) — обучение небольшого набора векторов, добавляемых перед эмбеддингами входных токенов. Параметры исходной модели остаются замороженными.

![Схема prompt tuning](https://i.imgur.com/VwNNKnb.png)


In [6]:
class WordEmbeddingsWithLearnedPrompts(nn.Module):
    """
    Для prompt tuning замените исходный слой эмбеддингов модели этим слоем.
    Он подставляет обучаемые векторы промпта вместо эмбеддингов первых num_prompts токенов.
    """

    def __init__(self, word_embeddings: nn.Embedding, num_prompts: int):
        super().__init__()
        self.original_word_embeddings = word_embeddings
        self.num_prompts = num_prompts
        self.learnable_prompts = nn.Parameter(
            torch.randn(1, num_prompts, word_embeddings.embedding_dim), requires_grad=True
        )

    def forward(self, input_ids: torch.LongTensor):
        # Форма input_ids: [batch_size, seq_length]
        assert input_ids.dtype == torch.int64
        assert input_ids.shape[1] > self.num_prompts
        assert torch.all(input_ids[:, :self.num_prompts] == tokenizer.pad_token_id).item(), (
            "Не забудьте добавить num_prompts PAD-токенов в начало input_ids"
        )

        # Получите эмбеддинги input_ids с помощью исходного слоя
        input_embeddings = self.original_word_embeddings(input_ids)  # Форма: [batch_size, seq_length, embedding_dim]

        # Замените эмбеддинги первых num_prompts токенов обучаемыми векторами промпта
        batch_size = input_ids.shape[0]
        learnable_prompts_expanded = self.learnable_prompts.expand(batch_size, -1, -1)  # Форма: [batch_size, num_prompts, embedding_dim]
        remaining_embeddings = input_embeddings[:, self.num_prompts:, :]  # Форма: [batch_size, seq_length - num_prompts, embedding_dim]

        # Объедините обучаемые векторы промпта с эмбеддингами оставшихся токенов
        output_embeddings = torch.cat([learnable_prompts_expanded, remaining_embeddings], dim=1)

        return output_embeddings


In [7]:
num_prompts = 16
test_emb_layer = WordEmbeddingsWithLearnedPrompts(model.model.embed_tokens, num_prompts=num_prompts).to(device)
test_input_ids = tokenizer("a cat say on a may", return_tensors='pt')['input_ids'].to(device)

space_for_prompts = torch.full([len(test_input_ids), num_prompts], fill_value=tokenizer.pad_token_id,
                               dtype=torch.int64, device=device)
test_inputs_with_prompts = torch.cat([space_for_prompts, test_input_ids], dim=1)

with torch.cuda.amp.autocast():
  test_prompt_embeddings = test_emb_layer(test_inputs_with_prompts)

assert test_prompt_embeddings.shape[:2] == test_inputs_with_prompts.shape
assert test_prompt_embeddings.shape[-1] == model.config.hidden_size
assert torch.allclose(test_prompt_embeddings[:, :num_prompts], test_emb_layer.learnable_prompts.float())
assert torch.allclose(test_prompt_embeddings[:, num_prompts:], model.model.embed_tokens(test_input_ids).float())
print("Проверки пройдены!")

Проверки пройдены!


/tmp/ipykernel_2400/1768851122.py:9: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


**Теперь, когда слой работает,** добавим обучаемые векторы промпта в основную модель и научим её рассказывать правду о лисе.


In [8]:
assert isinstance(model.model.embed_tokens, nn.Embedding), "Вы уже заменили слой эмбеддингов. Если новая реализация работает неверно, заново загрузите модель"

model.model.embed_tokens = WordEmbeddingsWithLearnedPrompts(model.model.embed_tokens, num_prompts=num_prompts).to(device)

opt = torch.optim.Adam([model.model.embed_tokens.learnable_prompts], lr=0.01)

In [9]:
the_truth = "A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it anyway!"
batch = tokenizer(the_truth, return_tensors='pt', return_token_type_ids=False).to(device)
space_for_prompts = torch.full([len(test_input_ids), num_prompts], fill_value=tokenizer.pad_token_id,
                               dtype=torch.int64, device=device)
batch['input_ids'] = torch.cat([space_for_prompts, batch['input_ids']], dim=1)
batch['attention_mask'] = torch.cat([torch.ones_like(space_for_prompts), batch['attention_mask']], dim=1)

outputs = model(**batch)
next_word_logits = outputs.logits[:, num_prompts : -1, :]
true_next_tokens = batch['input_ids'][:, num_prompts + 1:]
loss = F.cross_entropy(next_word_logits.flatten(0, 1), true_next_tokens.flatten(0, 1))
print("Потери:", loss)


num_epochs = 100 

for epoch in range(num_epochs):
    opt.zero_grad() 

    outputs = model(**batch)
    next_word_logits = outputs.logits[:, num_prompts : -1, :]
    true_next_tokens = batch['input_ids'][:, num_prompts + 1:]
    loss = F.cross_entropy(next_word_logits.flatten(0, 1), true_next_tokens.flatten(0, 1))
    loss.backward()

    opt.step()

    print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {loss.item():.4f}")

    if loss.item() < 0.1: # добавил остановку на всякий случай, дальше как раз такое значение проверяется 
        break
# raise NotImplemented("Задание: обучите векторы промпта в цикле, уменьшая потери с помощью оптимизатора opt")

Потери: tensor(7.8287, device='cuda:0', grad_fn=<NllLossBackward0>)
Epoch 1/100, Loss: 7.8287
Epoch 2/100, Loss: 7.0383
Epoch 3/100, Loss: 6.6210
Epoch 4/100, Loss: 6.2049
Epoch 5/100, Loss: 5.8268
Epoch 6/100, Loss: 5.4977
Epoch 7/100, Loss: 5.2023
Epoch 8/100, Loss: 4.9289
Epoch 9/100, Loss: 4.6740
Epoch 10/100, Loss: 4.4367
Epoch 11/100, Loss: 4.2151
Epoch 12/100, Loss: 4.0067
Epoch 13/100, Loss: 3.8089
Epoch 14/100, Loss: 3.6194
Epoch 15/100, Loss: 3.4348
Epoch 16/100, Loss: 3.2520
Epoch 17/100, Loss: 3.0698
Epoch 18/100, Loss: 2.8901
Epoch 19/100, Loss: 2.7157
Epoch 20/100, Loss: 2.5486
Epoch 21/100, Loss: 2.3887
Epoch 22/100, Loss: 2.2340
Epoch 23/100, Loss: 2.0830
Epoch 24/100, Loss: 1.9350
Epoch 25/100, Loss: 1.7905
Epoch 26/100, Loss: 1.6497
Epoch 27/100, Loss: 1.5109
Epoch 28/100, Loss: 1.3714
Epoch 29/100, Loss: 1.2310
Epoch 30/100, Loss: 1.0917
Epoch 31/100, Loss: 0.9585
Epoch 32/100, Loss: 0.8392
Epoch 33/100, Loss: 0.7384
Epoch 34/100, Loss: 0.6526
Epoch 35/100, Loss: 0.5

In [10]:
# Итоговая проверка значения потерь
assert loss.item() <= 0.1
print("Отлично!")

Отлично!


In [11]:
prompt = 'A quick brown fox'
batch = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False).to(device)
batch['input_ids'] = torch.cat([space_for_prompts, batch['input_ids']], dim=1)
batch['attention_mask'] = torch.cat([torch.ones_like(space_for_prompts), batch['attention_mask']], dim=1)


for i in range(15):
    next_token = model(**batch).logits[0, -1].argmax(-1).reshape(1, 1)
    batch['input_ids'] = torch.cat([batch['input_ids'], next_token], dim=-1)
    batch['attention_mask'] = torch.cat([batch['attention_mask'], torch.ones_like(next_token)], dim=-1)

print("\nРезультат:", tokenizer.decode(batch['input_ids'][0, num_prompts:].cpu().numpy().tolist()))

# Если всё сделано верно, модель будет отрицать, что лиса перепрыгнула через ленивую собаку


Результат: <s>A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it


### Используем Hugging Face PEFT (2 балла)

[`peft`](https://huggingface.co/docs/peft/index) — библиотека для дообучения предобученных трансформеров с небольшим числом обучаемых параметров (**P**arameter **E**fficient **F**ine-**T**uning). Она предоставляет общий интерфейс для prompt tuning, prefix tuning и нескольких методов на основе адаптеров.

Перед выполнением следующей ячейки заново загрузите исходную модель из начала ноутбука: здесь нужен исходный слой эмбеддингов, без вашей обёртки `WordEmbeddingsWithLearnedPrompts`.


In [13]:
import peft

manual_prompt_layer = model.model.embed_tokens
model.model.embed_tokens = manual_prompt_layer.original_word_embeddings

assert isinstance(model.model.embed_tokens, nn.Embedding), "Заново загрузите исходную модель"

peft_config = peft.PromptTuningConfig(task_type=peft.TaskType.CAUSAL_LM, num_virtual_tokens=16)
model = peft.get_peft_model(model, peft_config)  # Для большинства методов PEFT этот вызов также изменяет исходный объект model
print("Обучаемых параметров:", sum(p.numel() for p in model.parameters() if p.requires_grad))
print("Всего параметров (без учёта квантования):", sum(p.numel() for p in model.parameters()))

Обучаемых параметров: 65536
Всего параметров (без учёта квантования): 3500478464


In [ ]:
# Задание: обучите модель с обёрткой PEFT так, чтобы потери при предсказании следующего токена стали меньше 0.1
# PAD-токены вручную добавлять больше не нужно, но первые num_virtual_tokens позиций logits нужно пропустить
# В конце сгенерируйте продолжение фразы и убедитесь, что модель усвоила правду

In [ ]:
# Задаём целевую фразу
# the_truth = "A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it anyway!"
the_truth = "A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it anyway!"
batch = tokenizer(the_truth, return_tensors="pt", return_token_type_ids=False).to(device)

In [14]:
# Настройки обучения
loss_threshold = 0.1  # Целевой порог потерь
num_epochs = 100  # Максимальное число эпох
learning_rate = 0.01  # Скорость обучения

# Создайте оптимизатор для обучаемых параметров — векторов промпта PEFT
# optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
optimizer = torch.optim.Adam((p for p in model.parameters() if p.requires_grad), lr=learning_rate)


# Задаём целевую фразу
the_truth = "A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it anyway!"
batch = tokenizer(the_truth, return_tensors="pt", return_token_type_ids=False).to(device)

# Цикл обучения
for epoch in range(num_epochs):
    optimizer.zero_grad()
    # Прямой проход
    outputs = model(**batch)

    # Пропустите logits для виртуальных токенов и последнего токена
    next_word_logits = outputs.logits[:, num_prompts : -1, :]  # Пропускаем виртуальные токены
    true_next_tokens = batch['input_ids'][:, 1:]  # Сдвигаем целевые токены на одну позицию

    # Вычисляем потери
    loss = F.cross_entropy(
        next_word_logits.flatten(0, 1),
        true_next_tokens.flatten(0, 1)
    )

    # Обратное распространение и обновление параметров
    loss.backward()
    optimizer.step()

    # Выводим потери, чтобы отслеживать обучение
    print(f"Эпоха {epoch + 1}/{num_epochs}, потери: {loss.item()}")

    # Останавливаем обучение, если потери ниже порога
    if loss.item() < loss_threshold:
        print("Целевой порог потерь достигнут. Останавливаем обучение.")
        break
else:
    print("Достигнуто максимальное число эпох; целевой порог потерь ещё не достигнут.")

Эпоха 1/100, потери: 6.9511942863464355
Эпоха 2/100, потери: 6.449550151824951
Эпоха 3/100, потери: 5.9708709716796875
Эпоха 4/100, потери: 5.542173385620117
Эпоха 5/100, потери: 5.211521625518799
Эпоха 6/100, потери: 4.9333577156066895
Эпоха 7/100, потери: 4.683582782745361
Эпоха 8/100, потери: 4.458059310913086
Эпоха 9/100, потери: 4.253197193145752
Эпоха 10/100, потери: 4.061577320098877
Эпоха 11/100, потери: 3.873884677886963
Эпоха 12/100, потери: 3.68485951423645
Эпоха 13/100, потери: 3.494296073913574
Эпоха 14/100, потери: 3.3043100833892822
Эпоха 15/100, потери: 3.117095708847046
Эпоха 16/100, потери: 2.9344825744628906
Эпоха 17/100, потери: 2.7582573890686035
Эпоха 18/100, потери: 2.5897176265716553
Эпоха 19/100, потери: 2.428849935531616
Эпоха 20/100, потери: 2.274771213531494
Эпоха 21/100, потери: 2.1272003650665283
Эпоха 22/100, потери: 1.9865813255310059
Эпоха 23/100, потери: 1.8534599542617798
Эпоха 24/100, потери: 1.7283214330673218
Эпоха 25/100, потери: 1.610111951828003

In [15]:
# Итоговая проверка: потери должны быть ниже порога
assert loss.item() < loss_threshold, "Не удалось снизить потери ниже заданного порога."
print("Обучение завершено успешно! Потери ниже 0.1.")

Обучение завершено успешно! Потери ниже 0.1.


In [16]:
prompt = "A quick brown fox"
batch = tokenizer(prompt, return_tensors="pt", return_token_type_ids=False).to(device)

# Генерируем 15 токенов
for i in range(15):
    # Прямой проход для получения logits
    outputs = model(**batch)
    next_token = outputs.logits[0, -1].argmax(-1).reshape(1, 1)

    # Добавляем следующий токен к input_ids
    batch["input_ids"] = torch.cat([batch["input_ids"], next_token], dim=-1)

    # Дополняем attention_mask, чтобы её длина совпадала с новой длиной input_ids
    new_attention_mask = torch.ones_like(next_token, dtype=batch["attention_mask"].dtype).to(device)
    batch["attention_mask"] = torch.cat([batch["attention_mask"], new_attention_mask], dim=-1)

# Декодируем сгенерированную последовательность
# Здесь виртуальные токены добавляет PEFT внутри модели; в batch["input_ids"] их нет
decoded_output = tokenizer.decode(batch["input_ids"][0].cpu().numpy().tolist(), skip_special_tokens=True)
print("\nРезультат:", decoded_output)



Результат: A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it


### Дообучение с помощью LoRA (2 балла)

Для более содержательных задач можно использовать адаптеры низкого ранга из [статьи о LoRA](https://arxiv.org/pdf/2106.09685.pdf).

Основная идея — добавить обучаемый адаптер низкого ранга **параллельно существующему линейному слою** и сложить их выходы:
<center><img src="https://i.imgur.com/6bQLNiG.png" width=240px></center>

В исходной статье LoRA адаптеры добавляли только к матрицам проекций механизма внимания. Однако [последующие работы](https://arxiv.org/abs/2305.14314) показывают, что полезно адаптировать и полносвязные блоки (FFN). Прежде чем обучать модель, реализуйте базовый слой LoRA.


In [17]:
# Заново загружаем модель без ранее добавленных адаптеров PEFT
model = transformers.AutoModelForCausalLM.from_pretrained(
    model_name, device_map='auto', low_cpu_mem_usage=True, offload_state_dict=True,
    load_in_4bit=True, torch_dtype=torch.float32,  # Веса — 4-битные; нормализации и активации — в fp32
)
for param in model.parameters():
    param.requires_grad=False
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/33 [00:00<?, ?it/s]

In [20]:
class LoRALayer(nn.Module):
    """Добавляет LoRA-адаптер к существующему линейному слою."""
    def __init__(self, module: nn.Linear, rank: int):
        super().__init__()
        self.module = module  # Предобученный замороженный линейный слой
        self.adapter_A = nn.Parameter(torch.empty(module.in_features, rank, device=module.weight.device))
        nn.init.kaiming_uniform_(self.adapter_A, a=5 ** 0.5)
        self.adapter_B = nn.Parameter(torch.zeros(rank, module.out_features, device=module.weight.device))

    def forward(self, input):
        # Примените self.module и LoRA-адаптер к входу, затем верните сумму их выходов
        original_output = self.module(input)
        lora_output = (input @ self.adapter_A) @ self.adapter_B

        return original_output + lora_output

In [21]:
# Проверяем вашу реализацию
test_linear = nn.Linear(128, 128)
test_linear.weight.data[...] = torch.eye(128)
test_adapter = LoRALayer(test_linear, rank=8)

assert torch.allclose(test_adapter(torch.ones(1, 1, 128)), test_linear.bias + 1), "Проверьте реализацию прямого прохода"

test_adapter.adapter_A.data[...] = torch.linspace(0.1, -0.5, 128 * 8).view(128, 8)
test_adapter.adapter_B.data[...] = torch.linspace(0.5, -0.1, 128 * 8).view(8, 128)
test_linear.bias.data[...] = torch.linspace(1., -1., 128)

dummy_loss = F.mse_loss(test_adapter(torch.ones(1, 128) / 128).squeeze(), torch.linspace(-1, 1, 128))
assert torch.allclose(dummy_loss, torch.tensor(1.3711389), rtol=0, atol=1e-4)
dummy_loss.backward()
assert all(w.grad is not None for w in [test_adapter.adapter_A, test_adapter.adapter_B]), "Для некоторых параметров адаптера не вычислен градиент"
assert torch.allclose(test_adapter.adapter_A.grad.sum(), torch.tensor(-0.60158), rtol=0, atol=1e-4), "Неверный градиент по матрице A"
assert torch.allclose(test_adapter.adapter_B.grad.sum(), torch.tensor(0.9931), rtol=0, atol=1e-4), "Неверный градиент по матрице B"
# Неверный градиент: реализация отличается от формулы LoRA или нарушено вычисление градиентов (например, используется no_grad)
del dummy_loss, test_linear, test_adapter
print("Все проверки пройдены!")

Все проверки пройдены!


### Добавляем LoRA в модель

Код ниже добавляет LoRA-адаптеры к линейным проекциям Q/K/V в механизме внимания Llama. Можно также модифицировать другие слои:

* `self_attn.o_proj` — выходную проекцию механизма внимания;
* `mlp.up_proj`, `mlp.gate_proj`, `mlp.down_proj` — полносвязные слои трансформера;
* `lm_head` — выходной слой языковой модели.

Проверка в следующей ячейке ожидает 96 адаптеров: по три проекции Q/K/V в каждом из 32 блоков Llama-7B. Дополнительные адаптеры для итогового задания добавляйте после этой проверки.


In [22]:
lora_rank = 8

for name, module in model.model.layers.named_modules():
    if 'LlamaDecoderLayer' in repr(type(module)):
        module.self_attn.q_proj = LoRALayer(module.self_attn.q_proj, rank=lora_rank).to(device)
        module.self_attn.k_proj = LoRALayer(module.self_attn.k_proj, rank=lora_rank).to(device)
        module.self_attn.v_proj = LoRALayer(module.self_attn.v_proj, rank=lora_rank).to(device)

assert sum(isinstance(module, LoRALayer) for module in model.modules()) == 96  # Для Llama-7B

In [23]:
batch = tokenizer("This model wants to share its greatest secret:", return_tensors='pt', return_token_type_ids=False)
# Проверяем один шаг обучения: градиенты должны быть ненулевыми
with torch.cuda.amp.autocast(dtype=torch.float32):
    out = model.forward(**batch)
    (out.logits.norm() / 100).backward()

for i, module in enumerate(model.modules()):
    if isinstance(module, LoRALayer):
        assert module.adapter_B.grad is not None
        assert module.adapter_B.grad.norm().item() > 0

model.zero_grad(set_to_none=True)
print("Проверка градиентов пройдена, отлично!")

/tmp/ipykernel_2400/495728678.py:3: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.float32):


Проверка градиентов пройдена, отлично!


### Пример обучения модели

Пример ниже показывает, как обучать LoRA-адаптеры на небольшом демонстрационном датасете. В итоговом задании вам предстоит организовать похожее обучение.

**Обратите внимание:** условия итогового задания находятся ниже. Эта ячейка — только пример.


In [24]:
# Проверяем, способна ли модель обучаться. Для полноценного обучения измените max_steps
import datasets
data = datasets.load_dataset("Abirate/english_quotes", split="train[:32]") # 32 примера
data = data.map(lambda samples: tokenizer(samples['quote']), batched=True)
model._hf_peft_config_loaded = True  # Отключаем предупреждение Trainer из Hugging Face

trainer = transformers.Trainer(
    model=model, train_dataset=data,
    args=transformers.TrainingArguments(
        per_device_train_batch_size=2, 
        gradient_accumulation_steps=1, # Накопление градиентов увеличивает эффективный размер батча
        warmup_steps=250, 
        max_steps=100, 
        learning_rate=2e-4, 
        fp16=True,
        logging_steps=1, 
        output_dir='outputs', 
        report_to=None,
        save_strategy="no" # Для совместимости по состоянию на октябрь 2025 года (спасибо Владу!) 
    ),
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False)
)
# При предупреждениях о кэше задайте model.config.use_cache = False. Перед генерацией включите кэш обратно!

trainer.train()

# Это только пример: можно прервать его, не дожидаясь завершения обучения :)

README.md: 0.00B [00:00, ?B/s]

quotes.jsonl:   0%|          | 0.00/647k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2508 [00:00<?, ? examples/s]

Map:   0%|          | 0/32 [00:00<?, ? examples/s]

/home/uuu10/projects/hse/nlp-1-2026/.venv/lib/python3.12/site-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
max_steps is given, it will override any value given in num_train_epochs
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
1,1.272000
2,0.377500
3,1.475000
4,1.434000
5,0.860000
6,1.648300
7,1.839100
8,1.269800
9,0.560500
10,1.285200


TrainOutput(global_step=100, training_loss=0.5390028468519449, metrics={'train_runtime': 45.2475, 'train_samples_per_second': 4.42, 'train_steps_per_second': 2.21, 'total_flos': 634829603291136.0, 'train_loss': 0.5390028468519449, 'epoch': 6.25})

### Итоговое задание: обучаем модель на практике (5 баллов)

Ваша задача — дообучить модель **генерировать код на Python**. Используйте предыдущие примеры как основу.

* **Данные:** используйте [codeparrot-clean](https://huggingface.co/datasets/codeparrot/codeparrot-clean) или другой датасет с кодом на Python. Для этого упражнения достаточно небольшой части обучающей выборки.
* **Предобработка:** оставьте файлы с расширением `.py`. Для `codeparrot-clean` этот шаг можно пропустить: он уже содержит код на Python.
* **Длина примеров:** возьмите первые 512 символов каждого примера. Здесь ограничение задано в символах, а не в токенах.
* **Адаптеры:** используйте реализованную выше LoRA **и хотя бы одно** из следующих дополнений:
  - адаптер для `lm_head`;
  - адаптеры для компонентов MLP (`mlp.*`);
  - обучаемые входные эмбеддинги (потребуется скорректировать использование памяти).
* **Обучение:** обучать модель до сходимости не обязательно. Ориентир — 500 шагов: после этого проверьте, генерирует ли модель код методом `.generate()`. Используйте эффективный размер батча не меньше 4 и накопление градиентов `gradient_accumulation_steps=4`. Эффективный размер батча равен `per_device_train_batch_size × gradient_accumulation_steps × число GPU`.

В библиотеке `peft` тоже есть реализация LoRA. Однако для этого задания нужно показать **хотя бы один полный запуск обучения с вашей собственной реализацией LoRA**.

**Альтернативное задание:** вместо генерации кода на Python можно выбрать другой датасет и задачу, например тексты любимого исполнителя или расшифровки подкаста, если такое использование данных этично. Покажите на примерах, чему модель научилась — или не научилась, — сравнив генерации до и после дообучения, как в шаблоне ниже.


In [ ]:
from datasets import load_dataset

code_stream = load_dataset(
    "codeparrot/codeparrot-clean",
    split="train",
    streaming=True,
)

example = next(iter(code_stream))

print(code_data)

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/54 [00:00<?, ?it/s]

Поля: ['repo_name', 'path', 'copies', 'size', 'content', 'license', 'hash', 'line_mean', 'line_max', 'alpha_frac', 'autogenerated']
Путь: examples/asyncio/websocket/echo/client_coroutines.py
Начало файла:

###############################################################################
##
##  Copyright (C) 2013-2014 Tavendo GmbH
##
##  Licensed under the Apache License, Version 2.0 (the "License");
##  you may not use this file except in compliance with the License.
##  You may obtain a copy of the License at
##
##      http://www.apache.org/licenses/LICENSE-2.0
##
##  Unless required by applicable law or agreed to in writing, software
##  distributed under the License is distributed on an "AS IS" BASIS,
##  WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
##  See the License for the specific language governing permissions and
##  limitations under the License.
##
###############################################################################

from autobahn.async

In [ ]:
prompts =  ['', 'import', 'from', 'while', 'try', 'if', 'for', 'torch']  # Можно добавить другие промпты, в том числе не связанные напрямую с Python

# <A WHOLE LOT OF YOUR CODE>
# До дообучения сгенерируйте исходные примеры для выбранных промптов
# Используйте transformers.Trainer, как в примере выше, или собственный код обучения
# После обучения покажите генерации модели: ожидаются фрагменты кода на Python
# Для удобства сравнения оформите генерации в таблицу, например с помощью pandas или HTML
# Модель с LoRA генерирует текст тем же способом, что и исходная модель в примерах выше

@torch.inference_mode()
def generate_code(prompt, max_new_tokens=80):
    model.eval()

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        return_token_type_ids=False,
    ).to(device)

    generated_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        num_beams=1,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        use_cache=True,
    )

    input_length = 512  # длина входа по оси последовательности
    continuation_ids =   # первый пример, только новые токены

    return tokenizer.decode(
        ...,  # идентификаторы продолжения
        skip_special_tokens=True,
    )

In [ ]:
# Этот шаблон позволяет сравнить сгенерированные примеры кода в таблице
# Можно оформить результаты другим удобным способом

from IPython.display import HTML, display
table_template = """<table style="border:1px solid black" >
  <tr>
    <th style="text-align: center; border:1px solid black">ПРОМПТ</th>
    <th style="text-align: center; border:1px solid black">ДО ОБУЧЕНИЯ</th>
    <th style="text-align: center; border:1px solid black">ПОСЛЕ ОБУЧЕНИЯ</th>
  </tr>
{}
</table>"""

row_template = '''  <tr>
    <td style="width:20%; border:1px solid black"><pre align="left">`{}`</pre></td>
    <td style="width:40%; border:1px solid black"><pre align="left">{}</pre></td>
    <td style="width:40%; border:1px solid black"><pre align="left">{}</pre></td>
  </tr>'''

rows = []

for prompt in prompts:
    # Замените строки-заглушки в аргументах format() реальными генерациями
    rows.append(row_template.format(prompt, "ГЕНЕРАЦИЯ ДО ДООБУЧЕНИЯ", "ГЕНЕРАЦИЯ ПОСЛЕ ДООБУЧЕНИЯ"))

display(HTML(table_template.format('\n'.join(rows))))

PROMPT,BEFORE,AFTER
``,BEFORE FINETUNING,TO BE GENERATED AFTER FINETUNING
`import`,BEFORE FINETUNING,TO BE GENERATED AFTER FINETUNING
`from`,BEFORE FINETUNING,TO BE GENERATED AFTER FINETUNING
`while`,BEFORE FINETUNING,TO BE GENERATED AFTER FINETUNING
`try`,BEFORE FINETUNING,TO BE GENERATED AFTER FINETUNING
`if`,BEFORE FINETUNING,TO BE GENERATED AFTER FINETUNING
`for`,BEFORE FINETUNING,TO BE GENERATED AFTER FINETUNING
`torch`,BEFORE FINETUNING,TO BE GENERATED AFTER FINETUNING


Если вы дошли до этого места — поздравляем! Вы выполнили все основные задания.

**Дополнительное задание (бонус):** попробуйте реализовать обучение префиксов, передаваемых в механизм внимания через `model.forward(..., past_key_values=your_prompts)`. О вариантах обучения промптов и префиксов можно прочитать в статьях [1](https://arxiv.org/abs/2104.08691) и [2](https://arxiv.org/abs/2101.00190).

### Дополнительные материалы

* [Квантование после обучения](https://arxiv.org/abs/2208.07339)
* [Запуск больших моделей с помощью Accelerate](https://huggingface.co/docs/accelerate/package_reference/big_modeling)
* [Библиотека адаптеров AdapterHub](https://adapterhub.ml/)

### Запуск других моделей

Этот подход можно адаптировать к другим моделям сопоставимого размера, например [Falcon-7B](https://huggingface.co/tiiuae/falcon-7b), [OPT-6.7B](https://huggingface.co/facebook/opt-6.7b) или [BLOOM-7.1B](https://huggingface.co/bigscience/bloom-7b1). Потребуются небольшие изменения кода:

1. Измените имя модели в `AutoModelForCausalLM.from_pretrained()` и загрузите подходящий токенизатор через `AutoTokenizer`.
2. В коде prompt tuning замените `model.model.embed_tokens` на путь к слою эмбеддингов выбранной модели. Найти его поможет `print(model)`.
3. Адаптируйте код добавления LoRA-слоёв: названия блоков трансформера и их компонентов у других моделей отличаются.
